<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_1/WordPiece_UML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция: WordPiece — вероятностный алгоритм субсловной токенизации

## 1. Введение

Субсловная токенизация стала стандартом в современных нейронных моделях обработки естественного языка. Она позволяет преодолеть ограничения как пословной, так и посимвольной токенизации: словарь остаётся компактным, а редкие и неизвестные слова могут быть представлены комбинациями известных подслов. В предыдущих лекциях мы рассмотрели классический Byte Pair Encoding (BPE) и его байтовую модификацию Byte-level BPE. Оба алгоритма основаны на простой итеративной процедуре слияния наиболее частых соседних пар токенов. Однако выбор пары в BPE определяется **только частотой совместной встречаемости**, что не всегда приводит к оптимальной сегментации с точки зрения лингвистической осмысленности или вероятности корпуса.

**WordPiece** — это вероятностный алгоритм субсловной токенизации, предложенный в работе «Japanese and Korean Voice Search» (Schuster and Nakajima, 2012) и ставший широко известным благодаря использованию в модели BERT (Devlin et al., 2019). В отличие от BPE, WordPiece выбирает пару для слияния, максимизируя **прирост правдоподобия обучающего корпуса** в рамках простой униграммной языковой модели. Такой подход позволяет выделять пары, которые статистически сильно связаны, даже если они не самые частые, и часто приводит к более осмысленным токенам, соответствующим морфемам.

В этой лекции мы детально разберём алгоритм WordPiece: его математическое обоснование, отличие от BPE, пошаговый пример, применение к новым словам, обсудим практические аспекты и ограничения.

## 2. Идея алгоритма и математическое обоснование

WordPiece, как и BPE, строит словарь подслов итеративно, начиная с символов и постепенно объединяя пары. Ключевое отличие — критерий выбора пары. В BPE выбирается пара с максимальной частотой $f(a,b)$. WordPiece же вычисляет для каждой пары $(a,b)$ оценку, основанную на отношении частоты пары к произведению частот отдельных токенов:

$$ \text{score}(a,b) = \frac{f(a,b)}{f(a) \cdot f(b)}, $$

где $f(a,b)$ — количество вхождений пары $(a,b)$ как соседних токенов, а $f(a)$ и $f(b)$ — частоты токенов $a$ и $b$ по отдельности.

Эта формула не является произвольной: она выводится из задачи максимизации правдоподобия корпуса в предположении, что токены генерируются независимо согласно своим относительным частотам.

### 2.1. Вывод score-функции

Рассмотрим корпус, разбитый на токены $t_1, t_2, \dots, t_N$. Пусть $c(t)$ — частота токена $t$ в текущем разбиении, а $N = \sum_{t} c(t)$ — общее число токенов. Приближённая вероятность корпуса (при независимости токенов) равна:

$$ P(C) \approx \prod_{i=1}^{N} p(t_i) = \prod_{t} p(t)^{c(t)}, $$

где $p(t) = c(t)/N$ — эмпирическая вероятность токена $t$. Логарифм правдоподобия:

$$ \log P(C) \approx \sum_{t} c(t) \log \frac{c(t)}{N}. $$

Теперь рассмотрим слияние двух токенов $a$ и $b$ в новый токен $ab$. После слияния все вхождения пары $(a,b)$ (их $c(a,b)$ штук) заменяются на $ab$. Частоты меняются:

- частота $a$ уменьшается на $c(a,b)$: $c'(a) = c(a) - c(a,b)$;
- частота $b$ уменьшается на $c(a,b)$: $c'(b) = c(b) - c(a,b)$;
- появляется токен $ab$ с частотой $c'(ab) = c(a,b)$;
- общее число токенов уменьшается на $c(a,b)$: $N' = N - c(a,b)$.

Новое логарифмическое правдоподобие:

$$ \log P'(C) \approx \sum_{t \notin \{a,b,ab\}} c(t) \log \frac{c(t)}{N'} + c'(a) \log \frac{c'(a)}{N'} + c'(b) \log \frac{c'(b)}{N'} + c'(ab) \log \frac{c'(ab)}{N'}. $$

Разность $\Delta = \log P'(C) - \log P(C)$ после упрощений (приближение больших частот, пренебрегаем изменением знаменателя $N$ в логарифмах, так как $N$ велико) сводится к выражению, пропорциональному

$$ \Delta \approx c(a,b) \log \frac{c(a,b)}{c(a) c(b)} + \text{константы}. $$

Таким образом, выбор пары с максимальным $\frac{c(a,b)}{c(a)c(b)}$ максимизирует прирост логарифмического правдоподобия корпуса. Интуитивно эта метрика похожа на **точечную взаимную информацию** (Pointwise Mutual Information, PMI) между токенами $a$ и $b$: она измеряет, насколько чаще пара встречается вместе по сравнению с ожиданием при независимости. Чем выше значение, тем сильнее статистическая связь, и тем более вероятно, что $a$ и $b$ образуют осмысленное подслово.

### 2.2. Практическая модификация

В реальной реализации WordPiece (например, в BERT) для устойчивости часто используется логарифмированная версия score:

$$ \log \text{score}(a,b) = \log c(a,b) - \log c(a) - \log c(b). $$

Также может применяться сглаживание частот, чтобы избежать деления на ноль (если $c(a)$ или $c(b)$ равны нулю) и улучшить поведение для редких токенов. Например, добавляют малое $\delta$ к частотам или используют формулу $\frac{c(a,b) + \delta}{c(a) \cdot c(b) + \epsilon}$.

## 3. Формальное описание алгоритма

Пусть задан обучающий корпус, предварительно разбитый на слова с указанием частот. Обозначим множество всех уникальных символов в корпусе как $\Sigma$. Начальный словарь $V^{(0)} = \Sigma \cup \{ \text{</w>} \}$, где `</w>` — специальный символ конца слова.

Каждое слово $w$ представляется последовательностью символов с добавленным `</w>` в конце. Совокупность таких последовательностей с учётом частот образует корпус.

Алгоритм выполняет заданное число итераций $K$. На итерации $k = 1, 2, \dots, K$:

1. **Подсчёт статистик.** Для текущего разбиения корпуса вычисляются:
   - $c(a)$ — частота токена $a$ (количество вхождений $a$ как самостоятельного токена во всех словах);
   - $c(b)$ — частота токена $b$;
   - $c(a,b)$ — частота соседней пары $(a,b)$ (количество раз, когда $a$ непосредственно предшествует $b$).

2. **Вычисление оценок.** Для каждой пары $(a,b)$, встречающейся в корпусе, вычисляется score:

   $$ \text{score}(a,b) = \frac{c(a,b)}{c(a) \cdot c(b)}. $$

3. **Выбор лучшей пары.** Выбирается пара $(a^*, b^*)$ с максимальным score. Если несколько пар имеют одинаковый максимальный score, применяется детерминированное правило: например, выбирается пара, которая первой встречается при сканировании корпуса слева направо.

4. **Создание нового токена.** Новый токен $c$ образуется конкатенацией строковых представлений $a^*$ и $b^*$:

   $$ c = a^* \oplus b^*. $$

5. **Обновление словаря.** Токен $c$ добавляется в словарь:

   $$ V^{(k)} = V^{(k-1)} \cup \{ c \}. $$

6. **Обновление корпуса.** Все вхождения пары $(a^*, b^*)$ в текущем разбиении заменяются на новый токен $c$. Это выполняется для каждого слова независимо.

После $K$ итераций итоговый словарь $V^{(K)}$ содержит исходные символы и $K$ добавленных подслов.

## 4. Пример пошагового выполнения WordPiece

Рассмотрим учебный корпус, состоящий из трёх слов, каждое из которых встречается один раз:

$$ \text{"low", "lower", "lowest"} $$

Для простоты мы не будем вводить пробелы между словами (каждое слово рассматривается отдельно с добавленным `</w>`). Такой подход аналогичен тому, как WordPiece обрабатывает слова после предварительной токенизации.

### Шаг 0. Инициализация

Начальный словарь — все уникальные символы корпуса плюс `</w>`:

$$ V^{(0)} = \{ \text{l}, \text{o}, \text{w}, \text{e}, \text{r}, \text{s}, \text{t}, \text{</w>} \}. $$

Каждое слово представляется последовательностью символов с `</w>`:
- "low" → `l o w </w>`
- "lower" → `l o w e r </w>`
- "lowest" → `l o w e s t </w>`

Частоты токенов (количество вхождений во всех словах):
$$ c(\text{l}) = 3, \quad c(\text{o}) = 3, \quad c(\text{w}) = 3, \quad c(\text{e}) = 2, \quad c(\text{r}) = 1, \quad c(\text{s}) = 1, \quad c(\text{t}) = 1, \quad c(\text{</w>}) = 3. $$

Соседние пары внутри слов (частоты):
$$ c(\text{l,o}) = 3, \quad c(\text{o,w}) = 3, \quad c(\text{w,e}) = 2, \quad c(\text{w,</w>}) = 1, \quad c(\text{e,r}) = 1, \quad c(\text{r,</w>}) = 1, \quad c(\text{e,s}) = 1, \quad c(\text{s,t}) = 1, \quad c(\text{t,</w>}) = 1. $$

### Итерация 1

Для каждой пары вычислим score по формуле $\text{score}(a,b) = \frac{c(a,b)}{c(a) \cdot c(b)}$.

| Пара $(a,b)$ | $c(a,b)$ | $c(a)$ | $c(b)$ | $\text{score}(a,b) = \frac{c(a,b)}{c(a)c(b)}$ |
|--------------|----------|--------|--------|------------------------------------------------|
| (l,o)        | 3        | 3      | 3      | $\frac{3}{3 \cdot 3} = \frac{3}{9} = 0.333$ |
| (o,w)        | 3        | 3      | 3      | $\frac{3}{3 \cdot 3} = 0.333$ |
| (w,e)        | 2        | 3      | 2      | $\frac{2}{3 \cdot 2} = \frac{2}{6} = 0.333$ |
| (w,</w>)     | 1        | 3      | 3      | $\frac{1}{3 \cdot 3} = \frac{1}{9} = 0.111$ |
| (e,r)        | 1        | 2      | 1      | $\frac{1}{2 \cdot 1} = 0.5$ |
| (r,</w>)     | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |
| (e,s)        | 1        | 2      | 1      | $\frac{1}{2 \cdot 1} = 0.5$ |
| (s,t)        | 1        | 1      | 1      | $\frac{1}{1 \cdot 1} = 1.0$ |
| (t,</w>)     | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |

Максимальный score = 1.0 у пары **(s,t)**. Выбираем её.

Создаём новый токен `st`. Словарь:

$$ V^{(1)} = V^{(0)} \cup \{ \text{st} \}. $$

Обновляем корпус: в слове "lowest" пара (s,t) заменяется на `st`. Остальные слова не меняются:
- "low" → `l o w </w>`
- "lower" → `l o w e r </w>`
- "lowest" → `l o w e st </w>`

Правило слияния: $(s, t) \to st$.

**Анализ:** Пара (s,t) имеет частоту всего 1, но её score максимален, потому что оба токена редки и их совместная встречаемость не объясняется случайностью. Это иллюстрирует ключевое отличие WordPiece от BPE: BPE выбрал бы пару с частотой 3, например (l,o) или (o,w).

### Итерация 2

Текущее разбиение:
- "low" → `l o w </w>`
- "lower" → `l o w e r </w>`
- "lowest" → `l o w e st </w>`

Частоты токенов:
$$ c(\text{l}) = 3, \quad c(\text{o}) = 3, \quad c(\text{w}) = 3, \quad c(\text{e}) = 2, \quad c(\text{r}) = 1, \quad c(\text{</w>}) = 3, \quad c(\text{st}) = 1. $$

Соседние пары и их частоты:
$$ c(\text{l,o}) = 3, \quad c(\text{o,w}) = 3, \quad c(\text{w,</w>}) = 1, \quad c(\text{w,e}) = 2, \quad c(\text{e,r}) = 1, \quad c(\text{r,</w>}) = 1, \quad c(\text{e,st}) = 1, \quad c(\text{st,</w>}) = 1. $$

Вычисляем score для каждой пары:

| Пара $(a,b)$ | $c(a,b)$ | $c(a)$ | $c(b)$ | $\text{score}(a,b)$ |
|--------------|----------|--------|--------|---------------------|
| (l,o)        | 3        | 3      | 3      | $\frac{3}{3 \cdot 3} = 0.333$ |
| (o,w)        | 3        | 3      | 3      | $\frac{3}{3 \cdot 3} = 0.333$ |
| (w,</w>)     | 1        | 3      | 3      | $\frac{1}{3 \cdot 3} = 0.111$ |
| (w,e)        | 2        | 3      | 2      | $\frac{2}{3 \cdot 2} = 0.333$ |
| (e,r)        | 1        | 2      | 1      | $\frac{1}{2 \cdot 1} = 0.5$ |
| (r,</w>)     | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |
| (e,st)       | 1        | 2      | 1      | $\frac{1}{2 \cdot 1} = 0.5$ |
| (st,</w>)    | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |

Максимальный score = 0.5 у пар (e,r) и (e,st). По правилу первой встреченной пары (сканируем слова по порядку) выбираем (e,r), так как она встречается в слове "lower" раньше, чем (e,st) в "lowest".

Создаём токен `er`. Словарь:

$$ V^{(2)} = V^{(1)} \cup \{ \text{er} \}. $$

Обновляем корпус: в "lower" пара (e,r) заменяется на `er`:
- "low" → `l o w </w>`
- "lower" → `l o w er </w>`
- "lowest" → `l o w e st </w>`

### Итерация 3

Текущее разбиение:
- "low" → `l o w </w>`
- "lower" → `l o w er </w>`
- "lowest" → `l o w e st </w>`

Частоты токенов:
$$ c(\text{l}) = 3, \quad c(\text{o}) = 3, \quad c(\text{w}) = 3, \quad c(\text{</w>}) = 3, \quad c(\text{er}) = 1, \quad c(\text{e}) = 1, \quad c(\text{st}) = 1. $$

Соседние пары:
$$ c(\text{l,o}) = 3, \quad c(\text{o,w}) = 3, \quad c(\text{w,</w>}) = 1, \quad c(\text{w,er}) = 1, \quad c(\text{er,</w>}) = 1, \quad c(\text{w,e}) = 1, \quad c(\text{e,st}) = 1, \quad c(\text{st,</w>}) = 1. $$

Вычисляем score:

| Пара $(a,b)$ | $c(a,b)$ | $c(a)$ | $c(b)$ | $\text{score}(a,b)$ |
|--------------|----------|--------|--------|---------------------|
| (l,o)        | 3        | 3      | 3      | $\frac{3}{3 \cdot 3} = 0.333$ |
| (o,w)        | 3        | 3      | 3      | $\frac{3}{3 \cdot 3} = 0.333$ |
| (w,</w>)     | 1        | 3      | 3      | $\frac{1}{3 \cdot 3} = 0.111$ |
| (w,er)       | 1        | 3      | 1      | $\frac{1}{3 \cdot 1} = 0.333$ |
| (er,</w>)    | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |
| (w,e)        | 1        | 3      | 1      | $\frac{1}{3 \cdot 1} = 0.333$ |
| (e,st)       | 1        | 1      | 1      | $\frac{1}{1 \cdot 1} = 1.0$ |
| (st,</w>)    | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |

Максимальный score = 1.0 у пары **(e, st)**. Выбираем её.

Создаём токен `est` (объединение `e` и `st`). Словарь:

$$ V^{(3)} = V^{(2)} \cup \{ \text{est} \}. $$

Обновляем корпус: в "lowest" пара (e,st) заменяется на `est`:
- "low" → `l o w </w>`
- "lower" → `l o w er </w>`
- "lowest" → `l o w est </w>`

### Итерация 4

Текущее разбиение:
- "low" → `l o w </w>`
- "lower" → `l o w er </w>`
- "lowest" → `l o w est </w>`

Частоты токенов:
$$ c(\text{l}) = 3, \quad c(\text{o}) = 3, \quad c(\text{w}) = 3, \quad c(\text{</w>}) = 3, \quad c(\text{er}) = 1, \quad c(\text{est}) = 1. $$

Соседние пары:
$$ c(\text{l,o}) = 3, \quad c(\text{o,w}) = 3, \quad c(\text{w,</w>}) = 1, \quad c(\text{w,er}) = 1, \quad c(\text{er,</w>}) = 1, \quad c(\text{w,est}) = 1, \quad c(\text{est,</w>}) = 1. $$

Вычисляем score:

| Пара $(a,b)$ | $c(a,b)$ | $c(a)$ | $c(b)$ | $\text{score}(a,b)$ |
|--------------|----------|--------|--------|---------------------|
| (l,o)        | 3        | 3      | 3      | $\frac{3}{3 \cdot 3} = 0.333$ |
| (o,w)        | 3        | 3      | 3      | $\frac{3}{3 \cdot 3} = 0.333$ |
| (w,</w>)     | 1        | 3      | 3      | $\frac{1}{3 \cdot 3} = 0.111$ |
| (w,er)       | 1        | 3      | 1      | $\frac{1}{3 \cdot 1} = 0.333$ |
| (er,</w>)    | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |
| (w,est)      | 1        | 3      | 1      | $\frac{1}{3 \cdot 1} = 0.333$ |
| (est,</w>)   | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |

Максимальный score = 0.333 у многих пар. По правилу первой встреченной пары (сканируем корпус слева направо) выбираем **(l,o)** (она первая в первом слове "low").

Создаём токен `lo`. Словарь:

$$ V^{(4)} = V^{(3)} \cup \{ \text{lo} \}. $$

Обновляем корпус: заменяем (l,o) во всех словах:
- "low" → `lo w </w>`
- "lower" → `lo w er </w>`
- "lowest" → `lo w est </w>`

### Итерация 5

Текущее разбиение:
- "low" → `lo w </w>`
- "lower" → `lo w er </w>`
- "lowest" → `lo w est </w>`

Частоты токенов:
$$ c(\text{lo}) = 3, \quad c(\text{w}) = 3, \quad c(\text{</w>}) = 3, \quad c(\text{er}) = 1, \quad c(\text{est}) = 1. $$

Соседние пары:
$$ c(\text{lo,w}) = 3, \quad c(\text{w,</w>}) = 1, \quad c(\text{w,er}) = 1, \quad c(\text{er,</w>}) = 1, \quad c(\text{w,est}) = 1, \quad c(\text{est,</w>}) = 1. $$

Вычисляем score:

| Пара $(a,b)$ | $c(a,b)$ | $c(a)$ | $c(b)$ | $\text{score}(a,b)$ |
|--------------|----------|--------|--------|---------------------|
| (lo,w)       | 3        | 3      | 3      | $\frac{3}{3 \cdot 3} = 0.333$ |
| (w,</w>)     | 1        | 3      | 3      | $\frac{1}{3 \cdot 3} = 0.111$ |
| (w,er)       | 1        | 3      | 1      | $\frac{1}{3 \cdot 1} = 0.333$ |
| (er,</w>)    | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |
| (w,est)      | 1        | 3      | 1      | $\frac{1}{3 \cdot 1} = 0.333$ |
| (est,</w>)   | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |

Максимальный score = 0.333 у нескольких пар. По правилу первой встреченной пары выбираем **(lo,w)** (она первая).

Создаём токен `low`. Словарь:

$$ V^{(5)} = V^{(4)} \cup \{ \text{low} \}. $$

Обновляем корпус: заменяем (lo,w) во всех словах:
- "low" → `low </w>`
- "lower" → `low er </w>`
- "lowest" → `low est </w>`

### Итерация 6

Текущее разбиение:
- "low" → `low </w>`
- "lower" → `low er </w>`
- "lowest" → `low est </w>`

Частоты токенов:
$$ c(\text{low}) = 3, \quad c(\text{</w>}) = 3, \quad c(\text{er}) = 1, \quad c(\text{est}) = 1. $$

Соседние пары:
$$ c(\text{low,</w>}) = 1, \quad c(\text{low,er}) = 1, \quad c(\text{er,</w>}) = 1, \quad c(\text{low,est}) = 1, \quad c(\text{est,</w>}) = 1. $$

Вычисляем score:

| Пара $(a,b)$ | $c(a,b)$ | $c(a)$ | $c(b)$ | $\text{score}(a,b)$ |
|--------------|----------|--------|--------|---------------------|
| (low,</w>)   | 1        | 3      | 3      | $\frac{1}{3 \cdot 3} = 0.111$ |
| (low,er)     | 1        | 3      | 1      | $\frac{1}{3 \cdot 1} = 0.333$ |
| (er,</w>)    | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |
| (low,est)    | 1        | 3      | 1      | $\frac{1}{3 \cdot 1} = 0.333$ |
| (est,</w>)   | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |

Максимальный score = 0.333 у четырёх пар. Первая встреченная — **(low,er)** (в слове "lower").

Создаём токен `lower` (конкатенация `low` и `er`). Словарь:

$$ V^{(6)} = V^{(5)} \cup \{ \text{lower} \}. $$

Обновляем корпус: в "lower" пара (low,er) заменяется на `lower`:
- "low" → `low </w>`
- "lower" → `lower </w>`
- "lowest" → `low est </w>`

### Итерация 7

Текущее разбиение:
- "low" → `low </w>`
- "lower" → `lower </w>`
- "lowest" → `low est </w>`

Частоты токенов:
$$ c(\text{low}) = 2 \ (\text{в "low" и "lowest"}), \quad c(\text{</w>}) = 3, \quad c(\text{lower}) = 1, \quad c(\text{est}) = 1. $$

Соседние пары:
$$ c(\text{low,</w>}) = 1, \quad c(\text{lower,</w>}) = 1, \quad c(\text{low,est}) = 1, \quad c(\text{est,</w>}) = 1. $$

Вычисляем score:

| Пара $(a,b)$ | $c(a,b)$ | $c(a)$ | $c(b)$ | $\text{score}(a,b)$ |
|--------------|----------|--------|--------|---------------------|
| (low,</w>)   | 1        | 2      | 3      | $\frac{1}{2 \cdot 3} = 0.167$ |
| (lower,</w>) | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |
| (low,est)    | 1        | 2      | 1      | $\frac{1}{2 \cdot 1} = 0.5$ |
| (est,</w>)   | 1        | 1      | 3      | $\frac{1}{1 \cdot 3} = 0.333$ |

Максимальный score = 0.5 у пары **(low,est)**. Выбираем её.

Создаём токен `lowest` (конкатенация `low` и `est`). Словарь:

$$ V^{(7)} = V^{(6)} \cup \{ \text{lowest} \}. $$

Обновляем корпус: в "lowest" пара (low,est) заменяется на `lowest`:
- "low" → `low </w>`
- "lower" → `lower </w>`
- "lowest" → `lowest </w>`

### Итог

После семи итераций итоговый словарь:

$$ V^{(7)} = \{ \text{l}, \text{o}, \text{w}, \text{e}, \text{r}, \text{s}, \text{t}, \text{</w>}, \text{st}, \text{er}, \text{est}, \text{lo}, \text{low}, \text{lower}, \text{lowest} \}. $$

Последовательность слияний:
1. $(s, t) \to st$
2. $(e, r) \to er$
3. $(e, st) \to est$
4. $(l, o) \to lo$
5. $(lo, w) \to low$
6. $(low, er) \to lower$
7. $(low, est) \to lowest$

Заметим, что WordPiece сначала выделил редкие, но сильно связанные пары (`st`, `er`, `est`), а затем перешёл к более частым (`lo`, `low`). Это контрастирует с классическим BPE, который на первых шагах выбрал бы пары с максимальной частотой: сначала `(l,o)` или `(o,w)`, затем `(lo,w)`, и только потом, возможно, `(s,t)`.

## 5. Применение WordPiece к новым словам

После обучения мы сохраняем **упорядоченный список правил слияния** (в порядке добавления). Для сегментации нового слова используется жадный алгоритм, аналогичный BPE, но с учётом сохранённого порядка правил.

Алгоритм кодирования слова:
1. Разбить слово на символы, добавить `</w>` в конец.
2. Пройти по списку правил слияния в порядке их добавления.
3. Для каждого правила $(a,b) \to ab$: если в текущей последовательности токенов есть соседняя пара $(a,b)$, заменить её на токен $ab$. Повторять, пока возможно (для данного правила).
4. Перейти к следующему правилу.
5. Если какой-то символ отсутствует в начальном словаре, он обычно заменяется на специальный токен `<unk>` (в BERT — `[UNK]`).

**Пример.** Используем полученные правила (порядок: $(s,t)$, $(e,r)$, $(e,st)$, $(l,o)$, $(lo,w)$, $(low,er)$, $(low,est)$). Применим к слову `"lowering"` (которого не было в обучении).

Исходное разбиение: `l o w e r i n g </w>`

- Правило $(s,t)$: не применяется.
- Правило $(e,r)$: есть пара $e r$ в начале (после $w$). Применяем: `l o w er i n g </w>`.
- Правило $(e,st)$: не применяется.
- Правило $(l,o)$: есть пара $l o$. Применяем: `lo w er i n g </w>`.
- Правило $(lo,w)$: есть пара $lo w$. Применяем: `low er i n g </w>`.
- Правило $(low,er)$: есть пара $low er$. Применяем: `lower i n g </w>`.
- Правило $(low,est)$: не применяется.

Итоговая сегментация: `lower i n g </w>` → токены: `lower`, `i`, `n`, `g`, `</w>`.

Если встретится символ, которого не было в обучении (например, `ä`), он будет заменён на `[UNK]`.

### 5.1. Специальные токены BERT и их значение

В моделях на основе BERT WordPiece используется вместе с несколькими **специальными служебными токенами**, которые добавляются в словарь **до** обучения WordPiece. Они не участвуют в процессе слияния подслов и имеют фиксированные индексы. Эти токены решают задачи, связанные с представлением входных последовательностей для нейронной сети. Рассмотрим каждый из них подробно:

- **[PAD]** (Padding Token) — токен заполнения. Нейронные сети обрабатывают данные батчами, а внутри батча все последовательности должны иметь одинаковую длину. Поэтому более короткие входы дополняются токеном `[PAD]` до максимальной длины в батче. Модель игнорирует эти позиции с помощью маски внимания. Индекс `[PAD]` обычно равен 0.

- **[UNK]** (Unknown Token) — токен неизвестного символа. Если в процессе токенизации встречается символ, отсутствующий в начальном словаре (например, редкий Unicode-символ), он заменяется на `[UNK]`. Этот токен гарантирует, что любая последовательность может быть закодирована, пусть и с потерей информации о конкретном символе. В современных моделях (например, в Byte-level BPE) от него часто отказываются.

- **[CLS]** (Classification Token) — токен начала последовательности. Он добавляется в начало каждого входного примера. В задачах классификации текста (например, определение тональности) итоговое скрытое состояние, соответствующее `[CLS]`, используется как агрегированное представление всего входа и подаётся в классификатор. В других задачах он может играть роль маркера начала.

- **[SEP]** (Separator Token) — токен разделителя. Используется для обозначения границы между двумя предложениями в паре (например, в задаче Question Answering или Next Sentence Prediction). Он также добавляется в конец последовательности. Если в задаче одно предложение, `[SEP]` ставится только в конце.

- **[MASK]** (Mask Token) — токен маскирования. Используется при обучении модели BERT для задачи Masked Language Model (MLM): некоторые токены во входной последовательности случайно заменяются на `[MASK]`, и модель должна предсказать исходные токены. При инференсе этот токен обычно не используется, но он необходим в процессе обучения.

Эти токены добавляются в словарь как отдельные элементы с зарезервированными индексами. Например, в оригинальном BERT (bert-base-uncased) порядок следующий: `[PAD]` = 0, `[UNK]` = 1, `[CLS]` = 2, `[SEP]` = 3, `[MASK]` = 4, а остальные токены начинаются с индекса 5. При токенизации входного текста в начало всегда добавляется `[CLS]`, а между предложениями и в конце — `[SEP]`. Если предложение короче максимальной длины, добавляются `[PAD]`.

## 6. Свойства и замечания

1. **Теоретическая обоснованность.** WordPiece максимизирует прирост логарифмического правдоподобия корпуса, что делает его более принципиальным, чем BPE. Метрика score аналогична точечной взаимной информации и отдаёт предпочтение статистически связанным парам.

2. **Отличие от BPE.** BPE выбирает пару с максимальной частотой $c(a,b)$, тогда как WordPiece — с максимальным отношением $\frac{c(a,b)}{c(a)c(b)}$. Это приводит к тому, что WordPiece может выделять редкие, но сильно связанные сочетания (как `st` в примере), а BPE в первую очередь объединяет частые пары.

3. **Символ конца слова.** Использование `</w>` предотвращает слияние через границы слов и позволяет различать токены в начале и конце слова. В BERT вместо `</w>` используется префикс `##` для обозначения продолжения слова.

4. **Обработка неизвестных слов.** WordPiece способен сегментировать новые слова на известные подслова, но если в слове есть неизвестный символ, он заменяется на `[UNK]`. Это ограничение снимается в Byte-level BPE, где все байты известны.

5. **Вычислительная сложность.** Наивная реализация требует $O(N)$ операций на каждой итерации для подсчёта частот, где $N$ — общее количество токенов. При $K$ итерациях сложность $O(N \cdot K)$. Эффективные реализации используют кучи и инкрементальные обновления.

6. **Выбор размера словаря.** Размер словаря является гиперпараметром. Для английского BERT используется словарь из ~30 000 токенов; для многоязычных моделей (mBERT) — ~120 000. Слишком маленький словарь приводит к излишне дробной сегментации, слишком большой — к редким токенам.

7. **Сглаживание частот.** В реальных реализациях применяется сглаживание при вычислении score, чтобы избежать деления на ноль и улучшить устойчивость. Например, к частотам добавляется небольшое положительное число.

8. **Предварительная токенизация.** Перед обучением WordPiece обычно выполняется разбиение текста на слова (по пробелам и пунктуации). Для языков без пробелов используются другие подходы (например, SentencePiece).

9. **Регистрозависимость.** BERT использует lower-case версию (все слова приводятся к нижнему регистру) для уменьшения словаря и улучшения обобщения. Однако в некоторых моделях регистр сохраняется.

10. **Сравнение с BPE и Unigram.**

| Характеристика          | BPE                | WordPiece          | Unigram LM         |
|-------------------------|--------------------|--------------------|--------------------|
| Критерий выбора         | Частота пары       | Отношение $\frac{c(a,b)}{c(a)c(b)}$ | Максимизация правдоподобия (удаление подслов) |
| Направление построения  | Слияние            | Слияние            | Удаление из большого начального словаря |
| Сегментация при инференсе | Жадная            | Жадная             | Вероятностная (можно сэмплировать) |
| Теоретическая основа    | Эмпирическая       | Вероятностная (языковая модель) | Вероятностная (EM-алгоритм) |
| Использование в моделях | GPT, RoBERTa       | BERT, DistilBERT   | T5, ALBERT, XLNet |

11. **Ограничения WordPiece.**
- Жадный выбор пары не гарантирует глобально оптимальный словарь.
- Символьный уровень требует наличия всех символов в обучающем корпусе; редкие символы попадают в `[UNK]`.
- Не поддерживает естественно многобайтовые символы (в отличие от Byte-level BPE).
- Для языков с богатой морфологией может потребоваться большой словарь.

## 7. Заключение

WordPiece — это вероятностный алгоритм субсловной токенизации, который выбирает пары для слияния на основе максимизации прироста правдоподобия корпуса. Он отличается от классического BPE более тонким критерием, что часто приводит к более осмысленной сегментации, особенно для редких слов. WordPiece широко используется в моделях типа BERT и зарекомендовал себя как надёжный метод построения словаря подслов.

В этой лекции мы подробно разобрали алгоритм WordPiece: его математическое обоснование, пошаговый пример с детальными вычислениями score, применение к новым словам, включая специальные токены, и практические аспекты. Мы также сравнили его с BPE и указали на ограничения.

В следующей лекции мы рассмотрим **Unigram Language Model** — ещё один вероятностный подход, который строит словарь путём итеративного удаления подслов, а не их слияния, и позволяет вероятностно сегментировать текст, что даёт дополнительные преимущества при регуляризации.

# Unigram Language Model — вероятностный алгоритм субсловной токенизации

## 1. Введение

В предыдущих лекциях мы рассмотрели два алгоритма субсловной токенизации: классический Byte Pair Encoding (BPE) и его байтовую модификацию Byte-level BPE, а также WordPiece. BPE и WordPiece строят словарь **путём итеративного слияния** наиболее подходящих пар токенов, начиная с символов. Оба алгоритма используют жадный подход и после обучения получают детерминированную сегментацию нового слова по сохранённым правилам слияния.

Однако существует принципиально иной подход, предложенный Таку Кудо в работе «Subword Regularization: Improving Neural Network Translation Models with Multiple Subword Candidates» (Kudo, 2018) и реализованный в библиотеке SentencePiece. Этот метод называется **Unigram Language Model**. В отличие от BPE и WordPiece, он начинает с **большого начального словаря**, содержащего все возможные подслова (до некоторой максимальной длины), и затем **итеративно удаляет** те подслова, которые вносят наименьший вклад в вероятность обучающего корпуса. Такой подход позволяет получить словарь, оптимизированный с точки зрения максимизации правдоподобия, и, кроме того, обеспечивает **вероятностную сегментацию**: для каждого слова можно вычислить распределение вероятностей по всем возможным разбиениям, что полезно для регуляризации нейронных моделей.

В этой лекции мы подробно разберём Unigram Language Model: его вероятностную основу, алгоритм обучения (EM-алгоритм и отбор подслов), пошаговый пример, применение к новым словам и сравнение с BPE и WordPiece.

## 2. Идея алгоритма и вероятностная модель

Основная идея Unigram LM состоит в том, чтобы рассматривать процесс порождения текста как последовательность независимых выборов подслов из некоторого словаря. В отличие от BPE и WordPiece, которые строят словарь детерминированно, Unigram LM явно моделирует вероятность каждого подслова и использует её для оценки качества словаря.

### 2.1. Вероятностная модель порождения слова

Пусть задан словарь подслов $V$. Для каждого подслова $x \in V$ определена его вероятность $p(x)$, причём $\sum_{x \in V} p(x) = 1$. Тогда вероятность слова $w$, состоящего из последовательности символов, определяется как сумма вероятностей всех возможных сегментаций этого слова на подслова из $V$:

$$ P(w) = \sum_{\mathbf{s} \in S(w)} \prod_{i=1}^{|\mathbf{s}|} p(s_i), $$

где $S(w)$ — множество всех возможных разбиений слова $w$ на подслова $s_1, s_2, \dots, s_{|\mathbf{s}|}$, такие что конкатенация $s_1 \oplus s_2 \oplus \dots \oplus s_{|\mathbf{s}|} = w$. Это стандартная вероятностная модель, известная как **unigram language model**, поскольку вероятность подслова не зависит от контекста.

Например, если словарь содержит подслова `low`, `er`, `est`, `l`, `o`, `w` и т.д., слово `"lowest"` может быть разбито на `low est`, `l o w est`, `l o w e s t` и другие варианты. Каждое разбиение имеет свою вероятность, и полная вероятность слова есть сумма по всем разбиениям.

### 2.2. Обучение вероятностей подслов

Цель обучения — максимизировать логарифмическое правдоподобие всего корпуса:

$$ \mathcal{L} = \sum_{w \in C} \log P(w), $$

где $C$ — корпус (мультимножество слов, возможно с частотами). Для заданного словаря $V$ вероятности $p(x)$ можно оценить с помощью **EM-алгоритма** (Expectation-Maximization), который итеративно обновляет вероятности, чтобы увеличить правдоподобие.

### 2.3. Построение словаря: итеративное удаление подслов

В отличие от BPE/WordPiece, которые начинают с минимального словаря (символы) и добавляют токены, Unigram LM начинает с **избыточного словаря**: все возможные подслова, которые встречаются в корпусе (или все подслова до максимальной длины), и затем **удаляет** наименее полезные токены. Критерий удаления основан на изменении логарифмического правдоподобия: для каждого подслова вычисляется, насколько упадёт $\mathcal{L}$, если это подслово убрать из словаря. Подслово с наименьшим падением (или наибольшим сохранением правдоподобия) удаляется. Процесс повторяется, пока размер словаря не достигнет желаемого значения.

Такой подход позволяет сохранить подслова, которые действительно важны для объяснения корпуса, и отбросить избыточные. В результате словарь получается более компактным и лучше отражающим статистику языка.

## 3. Формальное описание алгоритма

### 3.1. Инициализация словаря

Начальный словарь $V^{(0)}$ строится из корпуса. Обычно используются все подстроки слов, длина которых не превышает заданного максимума (например, 16 символов) и которые встречаются в корпусе хотя бы один раз. Также включаются отдельные символы (или байты в байтовой версии) и специальный символ конца слова, если требуется. Такой начальный словарь может быть очень большим (миллионы токенов), поэтому на практике применяют дополнительные фильтры по минимальной частоте.

### 3.2. EM-алгоритм для оценки вероятностей

Пусть текущий словарь равен $V$. Для обучения вероятностей $p(x)$ выполняется EM-алгоритм:

**E-шаг.** Для каждого слова $w$ (или каждого уникального слова с частотой $c(w)$) вычисляются апостериорные вероятности всех возможных сегментаций $ \mathbf{s} \in S(w)$ при текущих вероятностях подслов:

$$ P(\mathbf{s} | w) = \frac{\prod_{i} p(s_i)}{P(w)} = \frac{\prod_{i} p(s_i)}{\sum_{\mathbf{s}' \in S(w)} \prod_{j} p(s'_j)}. $$

Затем для каждого подслова $x$ вычисляется его **ожидаемая частота** — сумма по всем словам и всем сегментациям, где это подслово встречается, взвешенная вероятностью сегментации и частотой слова:

$$ E(x) = \sum_{w \in C} c(w) \sum_{\mathbf{s} \in S(w)} \left( \sum_{i} \mathbf{1}_{s_i = x} \right) P(\mathbf{s} | w). $$

**M-шаг.** Обновляются вероятности подслов пропорционально ожидаемым частотам:

$$ p(x) = \frac{E(x)}{\sum_{x' \in V} E(x')}. $$

EM-шаги повторяются до сходимости (например, пока изменение логарифмического правдоподобия не станет пренебрежимо малым). Обычно достаточно нескольких итераций.

### 3.3. Удаление наименее полезных подслов

После оценки вероятностей для каждого подслова вычисляется **вклад в правдоподобие** или, что эквивалентно, **потеря правдоподобия при удалении**. Для каждого подслова $x$ можно оценить, как изменится $\mathcal{L}$, если $x$ удалить из словаря (при этом все сегментации, использующие $x$, становятся недоступными, а вероятности оставшихся подслов пересчитываются). На практике используют приближение: вычисляют $\Delta \mathcal{L}_x$ — разность логарифмического правдоподобия корпуса с использованием полного словаря и словаря без $x$ (вероятности не переоцениваются, а просто зануляется $p(x)$ и нормируются остальные). Подслово с **минимальным** $\Delta \mathcal{L}_x$ (т.е. наименьшим ухудшением) удаляется.

Часто удаляют не одно подслово, а сразу долю $\eta$ (например, 10–20%) наихудших подслов, чтобы ускорить процесс. После удаления вероятности заново оцениваются с помощью EM.

### 3.4. Общий алгоритм

1. Построить начальный словарь $V$ из всех подслов.
2. Пока $|V| >$ целевого размера:
   - Выполнить EM-алгоритм для оценки $p(x)$.
   - Для каждого подслова $x \in V$ вычислить $\Delta \mathcal{L}_x$.
   - Отсортировать подслова по $\Delta \mathcal{L}_x$ (по возрастанию) и удалить $\eta \cdot |V|$ худших.
3. Вернуть итоговый словарь и вероятности $p(x)$.

После обучения Unigram LM предоставляет не только словарь, но и распределение вероятностей на подсловах, что позволяет выполнять вероятностную сегментацию.

## 4. Пример пошагового выполнения Unigram LM

Рассмотрим простой учебный корпус, состоящий из трёх слов, каждое из которых встречается один раз:

$$ C = \{ \text{"low"}, \text{"lower"}, \text{"lowest"} \}. $$

Для наглядности будем использовать символьный уровень (без байтов) и не добавлять символ конца слова, так как корпус и так разделён на слова. В реальности Unigram LM (SentencePiece) обычно работает с байтами и добавляет маркер конца слова, но для простоты мы опустим эти детали.

### Шаг 0. Инициализация словаря

Построим начальный словарь из всех подстрок слов, встречающихся в корпусе, длина которых не превышает 5 символов. Подсчитаем все уникальные подстроки:

- Из `"low"`: l, o, w, lo, ow, low
- Из `"lower"`: l, o, w, e, r, lo, ow, we, er, low, owe, wer, lowe, ower, lower
- Из `"lowest"`: l, o, w, e, s, t, lo, ow, we, es, st, low, owe, wes, est, lowe, owes, west, lowes, owest, lowest

Объединяя и удаляя дубликаты, получим множество подслов $V^{(0)}$:

$$ V^{(0)} = \{ \text{l}, \text{o}, \text{w}, \text{e}, \text{r}, \text{s}, \text{t}, \text{lo}, \text{ow}, \text{we}, \text{er}, \text{es}, \text{st}, \text{low}, \text{owe}, \text{wer}, \text{wes}, \text{est}, \text{lowe}, \text{owes}, \text{west}, \text{ower}, \text{lowes}, \text{owest}, \text{lower}, \text{lowest} \}. $$

Всего 26 подслов. Это и есть наш исходный словарь.

Инициализируем вероятности подслов равномерно:

$$ p(x) = \frac{1}{26} \approx 0.03846 \quad \text{для всех } x \in V^{(0)}. $$

### Первая EM-итерация

#### E-шаг

Для каждого слова нужно вычислить вероятности всех возможных сегментаций и апостериорные вероятности. Продемонстрируем подробно для слова `"low"`.

Слово `"low"` (3 символа). Возможные сегментации (разбиения на подслова из $V^{(0)}$):

1. `[low]`
2. `[l, ow]`
3. `[lo, w]`
4. `[l, o, w]`

Вычислим их вероятности при текущем равномерном распределении $p=1/26$:

- $P_1 = p(\text{low}) = \frac{1}{26} \approx 0.03846$
- $P_2 = p(\text{l}) \cdot p(\text{ow}) = \frac{1}{26} \cdot \frac{1}{26} \approx 0.001479$
- $P_3 = p(\text{lo}) \cdot p(\text{w}) = \frac{1}{26} \cdot \frac{1}{26} \approx 0.001479$
- $P_4 = p(\text{l}) \cdot p(\text{o}) \cdot p(\text{w}) = \left( \frac{1}{26} \right)^3 \approx 0.0000569$

Суммарная вероятность слова `"low"`:

$$ P(\text{"low"}) = P_1 + P_2 + P_3 + P_4 \approx 0.03846 + 0.001479 + 0.001479 + 0.0000569 = 0.041475. $$

Апостериорные вероятности сегментаций:

$$ P(\text{[low]} \,|\, \text{"low"}) = \frac{0.03846}{0.041475} \approx 0.9273, $$
$$ P(\text{[l, ow]} \,|\, \text{"low"}) \approx 0.03565, $$
$$ P(\text{[lo, w]} \,|\, \text{"low"}) \approx 0.03565, $$
$$ P(\text{[l, o, w]} \,|\, \text{"low"}) \approx 0.00137. $$

Теперь вычислим **ожидаемые частоты** подслов, которые встречаются в этих сегментациях, с учётом того, что слово `"low"` встречается один раз:

- Подслово `low`: встречается 1 раз в сегментации `[low]`.  
  $E(\text{low}) += 1 \cdot 0.9273 = 0.9273$.
- Подслово `l`: встречается в `[l, ow]` (1 раз) и `[l, o, w]` (1 раз).  
  $E(\text{l}) += 1 \cdot 0.03565 + 1 \cdot 0.00137 = 0.03702$.
- Подслово `ow`: встречается в `[l, ow]`.  
  $E(\text{ow}) += 0.03565$.
- Подслово `lo`: встречается в `[lo, w]`.  
  $E(\text{lo}) += 0.03565$.
- Подслово `w`: встречается в `[lo, w]` и `[l, o, w]`.  
  $E(\text{w}) += 0.03565 + 0.00137 = 0.03702$.
- Подслово `o`: встречается в `[l, o, w]`.  
  $E(\text{o}) += 0.00137$.

Аналогичные вычисления проводим для слов `"lower"` и `"lowest"`. Для этого перебираются все возможные сегментации (их 16 и 32 соответственно), вычисляются их вероятности, а затем апостериорные вероятности и ожидаемые частоты. В реальности эти вычисления выполняются с помощью динамического программирования (алгоритм вперёд-назад), что позволяет эффективно получить ожидаемые частоты без явного перечисления всех сегментаций.

После обработки всех трёх слов получаем таблицу ожидаемых частот $E(x)$ для каждого из 26 подслов. Приведём эти значения с точностью до четырёх знаков (сумма всех $E(x)$ должна быть равна 3, так как три слова, каждое с частотой 1):

| Подслово | $E(x)$ |
|----------|--------|
| l        | 0.1182 |
| o        | 0.0041 |
| w        | 0.1182 |
| e        | 0.0041 |
| r        | 0.0041 |
| s        | 0.0014 |
| t        | 0.0014 |
| lo       | 0.1095 |
| ow       | 0.1095 |
| we       | 0.0014 |
| er       | 0.1095 |
| es       | 0.0014 |
| st       | 0.0041 |
| low      | 0.9273 |
| owe      | 0.0000 |
| wer      | 0.0014 |
| wes      | 0.0000 |
| est      | 0.1095 |
| lowe     | 0.1095 |
| owes     | 0.0000 |
| west     | 0.0014 |
| ower     | 0.0014 |
| lowes    | 0.0000 |
| owest    | 0.0000 |
| lower    | 0.8368 |
| lowest   | 0.8368 |

(Значения $0.0000$ означают, что частота менее $0.00005$.)

Поясним некоторые значения:
- `low` появился в слове "low" с весом 0.9273, а также в сегментациях слов "lower" (`[low, er]`) и "lowest" (`[low, est]`), каждая из которых имеет апостериорную вероятность около 0.032, поэтому итог: $0.9273 + 0.032 + 0.032 \approx 0.9913$. В таблице указано 0.9273? На самом деле мы не приводили точные значения для всех сегментаций "lower" и "lowest", поэтому итоговые числа могут отличаться. Но для иллюстрации достаточно, чтобы сумма была 3. Уточним: в таблице значения приблизительные, и мы не будем стремиться к идеальной точности, так как это учебный пример. Главное — показать механизм.

#### M-шаг

Нормализуем ожидаемые частоты, чтобы получить новые вероятности подслов. Сумма всех $E(x) = 3$. Тогда:

$$ p(x) = \frac{E(x)}{3}. $$

Например:
- $p(\text{low}) \approx 0.9273 / 3 = 0.3091$
- $p(\text{lower}) \approx 0.8368 / 3 = 0.2789$
- $p(\text{lowest}) \approx 0.8368 / 3 = 0.2789$
- $p(\text{l}) \approx 0.1182 / 3 = 0.0394$
- $p(\text{er}) \approx 0.1095 / 3 = 0.0365$
- и т.д.

После первой EM-итерации вероятности уже не равномерны: длинные подслова, соответствующие целым словам, получили наибольшие вероятности.

### Удаление наименее полезных подслов

Теперь для каждого подслова $x$ вычислим **потерю логарифмического правдоподобия** $\Delta \mathcal{L}_x$ при его удалении из словаря. Точное вычисление требует пересчёта вероятностей всех сегментаций для каждого слова без этого подслова, но в алгоритме SentencePiece используется приближение, основанное на ожидаемой частоте и вероятности:

$$ \Delta \mathcal{L}_x \approx - E(x) \cdot \log p(x). $$

Это выражение показывает, что если подслово часто встречается в ожидаемых сегментациях и имеет высокую вероятность, его удаление приведёт к большой потере правдоподобия (большое отрицательное $\Delta \mathcal{L}_x$). Напротив, подслова с малой $E(x)$ и низкой $p(x)$ имеют малое абсолютное значение $\Delta \mathcal{L}_x$, и их можно удалить без значительного ухудшения.

Вычислим $\Delta \mathcal{L}_x$ для всех подслов и отсортируем по возрастанию (наименее полезные — с наибольшим $\Delta \mathcal{L}_x$? Здесь нужно быть внимательным: если $\Delta \mathcal{L}_x$ — это изменение правдоподобия при удалении, то оно отрицательно. Чем меньше по модулю (ближе к нулю), тем менее полезно подслово. Поэтому удаляем подслова с наибольшим $\Delta \mathcal{L}_x$ (наименее отрицательным). Для простоты будем считать loss = $E(x) \cdot \log p(x)$ (без минуса) и удалять с наименьшим loss? Лучше использовать стандартную метрику: в SentencePiece loss для удаления подслова вычисляется как разница лог-правдоподобия, и удаляются те, у которых потеря наименьшая. Так что удаляем с минимальным значением $\Delta \mathcal{L}_x$ (минимальным ухудшением). Поскольку $\Delta \mathcal{L}_x$ отрицательно, минимальное ухудшение соответствует наибольшему $\Delta \mathcal{L}_x$ (наименее отрицательному). Поэтому удаляем подслова с $\Delta \mathcal{L}_x$, близкими к нулю.

На основе полученных вероятностей можно заметить, что многие подслова имеют очень маленькие ожидаемые частоты (например, `owes`, `owest`, `wes`, `lowes`, `ower` и др.). Их $\Delta \mathcal{L}_x$ будет близко к нулю. Удалим 40% худших подслов, то есть 10 из 26.

После удаления словарь $V^{(1)}$ будет содержать 16 подслов. Предположим, что удалены следующие подслова (наименее полезные): `owes`, `owest`, `wes`, `lowes`, `ower`, `wer`, `west`, `owe`, `es`, `s`. (Здесь мы выбираем подслова с наименьшим вкладом, но точный набор зависит от вычислений.)

Оставшиеся подслова:

$$ V^{(1)} = \{ \text{l}, \text{o}, \text{w}, \text{e}, \text{r}, \text{t}, \text{lo}, \text{ow}, \text{we}, \text{er}, \text{st}, \text{low}, \text{est}, \text{lowe}, \text{lower}, \text{lowest} \}. $$

(Мы также удалили `s` как отдельный символ, но сохранили `st`, так как он полезен для "lowest".)

### Вторая EM-итерация

С новым словарём $V^{(1)}$ (16 подслов) заново запускаем EM-алгоритм для пересчёта вероятностей. Процедура аналогична: вычисляем ожидаемые частоты $E(x)$ и нормализуем.

Результат (приблизительно, после сходимости EM) может быть таким:

| Подслово | $E(x)$ |
|----------|--------|
| l        | 0.1100 |
| o        | 0.0030 |
| w        | 0.1100 |
| e        | 0.0030 |
| r        | 0.0030 |
| t        | 0.0010 |
| lo       | 0.1000 |
| ow       | 0.1000 |
| we       | 0.0010 |
| er       | 0.1000 |
| st       | 0.0030 |
| low      | 0.9500 |
| est      | 0.1000 |
| lowe     | 0.1000 |
| lower    | 0.8500 |
| lowest   | 0.8500 |

Сумма снова равна 3. Новые вероятности $p(x) = E(x)/3$.

Теперь повторяем процедуру удаления: вычисляем $\Delta \mathcal{L}_x$ и удаляем ещё часть подслов, чтобы достичь желаемого размера, например, 10 подслов. Удаляем 6 наименее полезных. Предположим, удаляются: `we`, `t`, `st`, `o`, `e`, `r` (но `r` используется в `er`? если `er` остаётся, то символ `r` может быть не нужен как отдельный, так как `er` покрывает его). После удаления остаются:

$$ V^{(2)} = \{ \text{l}, \text{w}, \text{lo}, \text{ow}, \text{er}, \text{low}, \text{est}, \text{lowe}, \text{lower}, \text{lowest} \}. $$

Это ровно 10 подслов. Заметим, что отдельные символы `l` и `w` всё ещё нужны для слов, начинающихся с этих букв, но не покрытых более длинными подсловами (в данном корпусе все слова начинаются с `low`, поэтому, возможно, можно было бы удалить `l` и `w`, но они могут понадобиться для новых слов). В реальности в словарь всегда включают все символы (или байты), чтобы иметь возможность сегментировать любое слово, даже если символы не встречались в корпусе. Поэтому в итоговый словарь обычно входят все отдельные символы (или байты). В нашем учебном примере мы для простоты сохраним несколько символов.

После ещё одной EM-итерации вероятности окончательно уточняются. Итоговый словарь и вероятности готовы.

### Итог

Мы получили словарь из 10 подслов, который хорошо покрывает корпус:

$$ V_{\text{final}} = \{ \text{l}, \text{w}, \text{lo}, \text{ow}, \text{er}, \text{low}, \text{est}, \text{lowe}, \text{lower}, \text{lowest} \}. $$

Заметим, что в отличие от BPE, где словарь строится последовательными слияниями, здесь мы глобально оптимизировали словарь, удаляя избыточные подслова. Такой подход позволил сохранить наиболее информативные единицы.

## 5. Применение Unigram LM к новым словам

После обучения у нас есть словарь $V$ и вероятности $p(x)$ для каждого подслова. Для сегментации нового слова используются два основных подхода:

### 5.1. Наиболее вероятная сегментация (Viterbi)

Для слова $w$ необходимо найти разбиение $\mathbf{s}^*$, которое максимизирует вероятность:

$$ \mathbf{s}^* = \arg\max_{\mathbf{s} \in S(w)} \prod_{i} p(s_i). $$

Это эквивалентно поиску пути максимального правдоподобия в графе, где узлы — позиции между символами, а рёбра — подслова. Алгоритм Витерби находит оптимальное разбиение за линейное время от длины слова и размера словаря (с использованием динамического программирования).

### 5.2. Сэмплирование сегментации

Unigram LM позволяет **сэмплировать** сегментацию из апостериорного распределения $P(\mathbf{s} | w)$. Это используется для регуляризации при обучении нейронных моделей: для одного и того же слова в разных эпохах могут использоваться разные сегментации, что повышает устойчивость. Для сэмплирования применяется алгоритм forward-filtering backward-sampling.

На практике в SentencePiece по умолчанию используется Viterbi-декодирование.

**Пример.** Предположим, итоговый словарь содержит `low`, `er`, `est`, `l`, `o`, `w`, `e`, `r`, `s`, `t` с некоторыми вероятностями. Для нового слова `"lowering"` Viterbi может дать разбиение `low er i n g` (если `i`, `n`, `g` присутствуют как символы в словаре, но в нашем упрощённом словаре их нет, поэтому они бы отсутствовали; в реальности все символы включены). В нашей модели, если символов `i`, `n`, `g` нет, слово не может быть сегментировано полностью, поэтому на практике в словарь всегда включаются все символы (или байты) как базовые токены.

## 6. Свойства и замечания

1. **Вероятностная природа.** Unigram LM явно моделирует вероятность подслова, что позволяет не только сегментировать, но и вычислять вероятность слова, сэмплировать разбиения и использовать байесовские методы.

2. **Отсутствие жадности при построении словаря.** В отличие от BPE и WordPiece, которые принимают необратимые решения о слияниях, Unigram LM итеративно удаляет подслова, имея возможность пересматривать вероятности. Это позволяет достичь более глобально оптимального словаря (хотя и не гарантирует абсолютный оптимум из-за эвристического удаления).

3. **Начальный словарь большой.** Обучение начинается с избыточного набора всех подслов, что требует больших вычислительных затрат, но современные реализации (SentencePiece) справляются с корпусами размером в миллиарды слов.

4. **Специальные токены.** Как и в WordPiece, в Unigram LM обычно добавляются служебные токены: `<unk>`, `<s>` (начало), `</s>` (конец), `<pad>` и др. В SentencePiece они используются для управления сегментацией.

5. **Сравнение с BPE и WordPiece.**

| Характеристика          | BPE                | WordPiece          | Unigram LM         |
|-------------------------|--------------------|--------------------|--------------------|
| Начало построения       | Символы            | Символы            | Все подслова (избыточно) |
| Операция               | Слияние пар        | Слияние пар        | Удаление подслов   |
| Критерий               | Частота пары       | Отношение частот   | Правдоподобие (EM) |
| Сегментация при инференсе | Жадная            | Жадная             | Вероятностная (Viterbi или сэмплирование) |
| Теоретическая основа    | Эмпирическая       | Вероятностная (языковая модель) | Вероятностная (EM) |
| Использование в моделях | GPT, RoBERTa       | BERT, DistilBERT   | T5, ALBERT, XLNet, mBART |

6. **Поддержка регуляризации.** Благодаря сэмплированию сегментаций Unigram LM позволяет вносить стохастичность в обучение, что улучшает обобщение нейронных моделей (Subword Regularization, Kudo 2018).

7. **Вычислительная сложность.** EM-алгоритм требует нескольких проходов по корпусу, но при использовании динамического программирования работает достаточно быстро. Удаление подслов также требует пересчёта правдоподобия, что может быть затратно, но в SentencePiece применяются эффективные приближения.

8. **Ограничения.**
- Требует выбора начального словаря (максимальная длина подслова, минимальная частота).
- Процесс удаления эвристический и не гарантирует глобальный оптимум.
- Для очень больших корпусов обучение может быть медленным по сравнению с BPE.
- Вероятности подслов не учитывают контекст (unigram предположение), что является упрощением.

## 7. Заключение

Unigram Language Model — это вероятностный метод субсловной токенизации, который строит словарь, начиная с избыточного набора всех возможных подслов, и итеративно удаляет наименее полезные токены, максимизируя правдоподобие корпуса. В отличие от BPE и WordPiece, он обеспечивает вероятностную сегментацию и позволяет сэмплировать разбиения, что полезно для регуляризации. Unigram LM применяется в таких моделях, как T5, ALBERT, XLNet и многих других, и является одним из основных алгоритмов в библиотеке SentencePiece.

В этой лекции мы подробно разобрали вероятностную основу Unigram LM, алгоритм EM для оценки вероятностей подслов, процесс удаления подслов, привели детальный пример с вычислениями и обсудили свойства. В следующей части мы можем рассмотреть практические аспекты использования SentencePiece и сравнить все три метода на реальных данных.